In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


## inicio

In [15]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text
import pandas as pd

import numpy as np
from funciones import *
from funciones_spark import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

    

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [17]:
filename='SUBIT_ALFIN.csv'
df_lista=cargar_archivo_csv(spark,filename,';',True)



filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db2.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V3_SS (RED_CALL)db.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db2.csv'
df_validar_03=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202608_V5_SS_EXT (CAMPO)_db.csv'
df_validar_04=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar_01=df_validar_01.drop('TASA_MIN_DESCUENTO')
df_validar_02=df_validar_02.drop('TASA_MIN_DESCUENTO')
df_validar_01=df_validar_01.withColumn('tipo_archivo',F.lit('campo'))
df_validar_02=df_validar_02.withColumn('tipo_archivo',F.lit('campo'))
df_validar_03=df_validar_03.withColumn('tipo_archivo',F.lit('call'))
df_validar_04=df_validar_04.withColumn('tipo_archivo',F.lit('call'))

df_validar=df_validar_01.unionByName(df_validar_02).unionByName(df_validar_03).unionByName(df_validar_04)

filename='BASE_TARGET_20260801.txt'
df_base=cargar_archivo_csv_ruta(spark,filename,'|',True,ruta_alfin)
df_base=df_base.select('DNI')
df_base=df_base.withColumn('tipo_base_int',F.lit('en_base'))


def completar_dni(df):
    return df.withColumn(
        "DNI",
        F.lpad(F.col("DNI").cast("string"), 8, "0")
    )

df_lista_1 = completar_dni(df_lista_1)
df_base = completar_dni(df_base)
df_validar = completar_dni(df_validar)
df_lista_1=df_lista.join(df_validar,['DNI'],'inner')
df_lista_1=df_lista_1.join(df_base,['DNI'],'left')
df_lista_1=df_lista_1.withColumn('tipo_base_int',when(F.col('tipo_base_int').isNull(),F.lit('fuera_de_base'))
                                            .otherwise(F.col('tipo_base_int')))
df_lista_1=df_lista_1.dropDuplicates(['DNI'])



In [18]:
df_lista_1.count()

150

In [19]:
from pyspark.sql.window import Window

w = Window.partitionBy("dni").orderBy(F.col("dni").asc())

df_lista_2 = (
    df_lista_1
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [21]:
df_lista_pd=df_lista_2.toPandas()

In [8]:
df_lista_pd["agencia"] = df_lista_pd["agencia"].str.strip()

In [9]:
reemplazos = {
    "PC TACNA": "TACNA",
    "AREQ PAMPILLA": "AREQUIPA PAMPILLA",      # este realmente no cambia
    "AREQ CAYMA": "AREQUIPA CAYMA",
    "ENMANCIPACION": "EMANCIPACION",
    "SAN JUAN DE LURIG": "SAN JUAN DE LURIGANCHO",
    "TRUJ CENTRO": "TRUJILLO CENTRO",
    "TRUJ AMERICA": "TRUJILLO AMERICA",
    "PC HUANCAYO": "HUANCAYO",
    "PC HUARAZ": "HUARAZ",
    "PAITA": "SULLANA",
}

df_lista_pd["agencia"] = df_lista_pd["agencia"].replace(reemplazos)

In [10]:
query = f"""
select agencia_Formulario,agencia_correo  as agencia from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)

df_Age = (
    df_Age
    .drop_duplicates(
        subset="agencia_Formulario",
        keep="first"
    )
    .reset_index(drop=True)
)



In [11]:
df_lista_pd_01=df_lista_pd.merge(df_Age,on='agencia',how='left')

In [156]:
df_lista_pd_01.loc[
    df_lista_pd_01["agencia_Formulario"].isna(),
    "agencia"
].unique()

array([], dtype=object)

In [12]:
query = f"""
	SELECT dni_cliente as dni,estado  as estdo_formulario,
    DATE(fecha_creacion) as fecha_carga_formulario,
    DATE(fecha_envio) as fecha_envio_formulario 
    FROM Alice.prospectos_envio_alfin 
    where DATE(fecha_creacion)>='2026-08-01'
"""
df_formulario_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT dni_cliente as dni,estado as estdo_correo,
    DATE(fecha_registro) as fecha_carga_correo,
    DATE(fecha_envio) as fecha_envio_correo 
    FROM Alice.prospectos_correos_alfin 
    where DATE(fecha_registro)>='2026-08-01'
"""
df_correos_alfin = pd.read_sql(query, engine_mysql)


In [13]:
df_formulario_alfin["fecha_envio_formulario"] = pd.to_datetime(
    df_formulario_alfin["fecha_envio_formulario"]
)
df_formulario_alfin["fecha_carga_formulario"] = pd.to_datetime(
    df_formulario_alfin["fecha_carga_formulario"]
)

df_formulario_alfin = (
    df_formulario_alfin
    .sort_values(
        by=["fecha_envio_formulario", "fecha_carga_formulario"],
        ascending=[False, False],
        na_position="last"
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )

)
df_correos_alfin["fecha_envio_correo"] = pd.to_datetime(
    df_correos_alfin["fecha_envio_correo"]
)

df_correos_alfin["fecha_carga_correo"] = pd.to_datetime(
    df_correos_alfin["fecha_carga_correo"]
)

df_correos_alfin = (
    df_correos_alfin
    .sort_values(
        by=["fecha_envio_correo", "fecha_carga_correo"],
        ascending=[False, False],
        na_position="last"
    )
    .drop_duplicates(
        subset="dni",
        keep="first"
    )
)

In [14]:
df_lista_pd_02=df_lista_pd_01.merge(df_formulario_alfin,on='dni',how='left')
df_lista_pd_02=df_lista_pd_02.merge(df_correos_alfin,on='dni',how='left')



In [15]:
dni_duplicados = (
    df_lista_pd_02["dni"]
    .value_counts()
    .loc[lambda x: x > 1]
)
print(dni_duplicados.index.tolist())


[]


In [16]:

# Convertir a numérico
df_lista_pd_02["OFERTA_MAX"] = pd.to_numeric(df_lista_pd_02["OFERTA_MAX"], errors="coerce")

bins = [0, 5000, 10000, 15000, 20000, 25000, 30000, np.inf]

labels = [
    "01. [0 - 5,000)",
    "02. [5,000 - 10,000)",
    "03. [10,000 - 15,000)",
    "04. [15,000 - 20,000)",
    "05. [20,000 - 25,000)",
    "06. [25,000 - 30,000)",
    "07. >= 30,000"
]

df_lista_pd_02["RANGO_OFERTA"] = pd.cut(
    df_lista_pd_02["OFERTA_MAX"],
    bins=bins,
    labels=labels,
    right=False
)

In [8]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

In [9]:
print(df_target_desembolso.shape)
print(df_fugas.shape)
print(df_target_desembolso.columns.to_list())
print(df_fugas.columns.to_list())

(9, 2)
(1047, 2)
['DNI', 'CANAL']
['DNI', 'CANALVENTA']


In [10]:
df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


In [22]:
df_lista_pd.shape

(150, 42)

In [23]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())




C:\Users\DATA\AppData\Local\Temp\ipykernel_25200\1641553792.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [24]:
df_lista_pd = df_lista_pd[
    (~df_lista_pd["DNI"].isin(dni_retiro)) &
    (~df_lista_pd["DNI"].isin(dni_desembolso)) 
].copy()

In [25]:
df_lista_pd.shape

(0, 42)

In [14]:

ruta_archivo = os.path.join(ruta_csv, 'lista_alfin.csv')
df_lista_pd.to_csv(ruta_archivo,sep=';')

In [ ]:
DELETE t1
FROM Alice.prospectos_correos_alfin t1
INNER JOIN Alice.prospectos_correos_alfin t2
    ON t1.dni_cliente= t2.dni_cliente
   AND t1.id < t2.id
WHERE t1.fecha_registro >= '2026-08-06 00:00:00';



In [28]:
df_fugas_des=df_desembolso[df_desembolso['CANAL']=='OTROS'].copy()
df_fugas_des = set(df_desembolso['dni_cliente'].dropna())


In [29]:
list_dni = dni_retiro.union(df_fugas_des)
list_cel = cel_retiro

In [30]:
in_clause_dni = ",".join(f"'{x}'" for x in list_dni)
in_clause_cel = ",".join(f"'{x}'" for x in list_cel)

In [31]:
from sqlalchemy import text, bindparam

list_dni = list(dni_retiro | dni_desembolso)
list_cel = list(cel_retiro)

query = text("""
DELETE
FROM Alice.prospectos_correos_alfin
WHERE dni_cliente IN :dni
   OR celular IN :cel
""").bindparams(
    bindparam("dni", expanding=True),
    bindparam("cel", expanding=True)
)

with engine_mysql.begin() as conn:
    result = conn.execute(
        query,
        {
            "dni": list_dni,
            "cel": list_cel
        }
    )

print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 0


In [33]:
from sqlalchemy import text, bindparam

list_dni = list(dni_retiro | dni_desembolso)

query = text("""
DELETE
FROM Alice.prospectos_envio_alfin
WHERE dni_cliente IN :dni
""").bindparams(
    bindparam("dni", expanding=True)
)

with engine_mysql.begin() as conn:
    result = conn.execute(
        query,
        {
            "dni": list_dni
        }
    )

print("Filas eliminadas:", result.rowcount)

Filas eliminadas: 362


In [130]:
# df=df_subir.toPandas()
query = f"""
select agencia_Formulario,agencia_correo from Alice.agencias_alfin
"""
df_Age = pd.read_sql(query, engine_mysql)